# Balanced arXiv IMSTE experiments

This notebook uses a reproducible subset of the prepared high-dimensional feature data so configuration experiments finish quickly. The 2D layouts are refitted whenever their experiment cells run. The default is 32,000 unique papers, sampled evenly across the eight domains (4,000 per domain). The requested number per class is capped by the smallest available class. Balanced selection never duplicates papers; change `TARGET_PER_CLASS` or `SAMPLE_MODE` to control the experiment subset. For a 640,000-row random confirmation run, set `SAMPLE_MODE = "random"` and `TARGET_PER_CLASS = 80_000`; for quick balanced runs, keep the default 4,000 per domain.

UMAP is fitted only as a comparison reference. **Independent IMSTE runs use no UMAP coordinates or UMAP initialization.** An opt-in UMAP warm-start cell is reserved for a diagnostic comparison and is excluded from candidate configurations. IMSTE variants start independently from PCA, spectral, or random coordinates, then test multilevel refinement, optimizer and repulsion settings, and local-neighbor graph alternatives. The notebook reports neighborhood fidelity, domain purity, relative local radius, and runtime. Subset and model settings are included in cache identities.

In [ ]:
from pathlib import Path
import sys
import json
import time
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists() and (PROJECT_ROOT.parent / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "notebooks" / "src"))
from arxiv_dataset import load_arxiv_dataset
from arxiv_imste import select_arxiv_experiment_subset

DATA_DIR = PROJECT_ROOT / "data" / "arxiv"
papers, X_full, data_manifest = load_arxiv_dataset(DATA_DIR)

primary_category = papers["categories"].fillna("unknown").str.split().str[0]
archive = primary_category.str.split(".").str[0]
domain_names = {
    "cs": "computer science",
    "math": "mathematics",
    "eess": "engineering",
    "stat": "statistics",
    "q-bio": "biology",
    "q-fin": "finance",
    "econ": "economics",
}
domain = archive.map(domain_names).fillna("physics")
domain_order = domain.value_counts().index.tolist()
labels_full = pd.Categorical(domain, categories=domain_order)

# Change these controls to run a smaller, reproducible experiment.
SAMPLE_MODE = "balanced"  # "balanced", "random", or "full"
TARGET_PER_CLASS = 4_000
SAMPLE_RANDOM_STATE = int(data_manifest["random_state"])
if SAMPLE_MODE == "balanced":
    counts = np.unique(np.asarray(labels_full), return_counts=True)[1]
    AVAILABLE_PER_CLASS = int(counts.min())
    PER_CLASS = min(TARGET_PER_CLASS, AVAILABLE_PER_CLASS)
    SAMPLE_SIZE = len(counts) * PER_CLASS
    if PER_CLASS < TARGET_PER_CLASS:
        print(f"Balanced sample capped at {PER_CLASS:,} rows per class by the minority class size.")
else:
    SAMPLE_SIZE = TARGET_PER_CLASS * len(np.unique(np.asarray(labels_full)))
X_experiment, labels_experiment, experiment_data_id = select_arxiv_experiment_subset(
    X_full,
    labels_full,
    dataset_id=data_manifest["dataset_id"],
    mode=SAMPLE_MODE,
    sample_size=SAMPLE_SIZE,
    random_state=SAMPLE_RANDOM_STATE,
)
print(f"Source data: {len(X_full):,} papers × {X_full.shape[1]:,} features")
print(f"Experiment subset: {len(X_experiment):,} rows ({SAMPLE_MODE}), ID {experiment_data_id}")
print(pd.Series(np.asarray(labels_experiment)).value_counts().sort_index().to_string())

# Shared geometry settings. Keep these fixed when comparing initializations.
RANDOM_STATE = int(data_manifest["random_state"])
METRIC = "euclidean"
N_NEIGHBORS = 15
UMAP_EPOCHS = 200
IMSTE_N_MSTS = 5
IMSTE_N_EPOCHS = 100
IMSTE_BATCH_SIZE = 32768
IMSTE_LEARNING_RATE = 0.05
IMSTE_NEGATIVE_RATIO = 1
IMSTE_LAMBDA_REP = 0.05
IMSTE_LOGISTIC_MARGIN = 0.25
IMSTE_LOGISTIC_TEMPERATURE = 0.5
IMSTE_DEVICE = "mps"  # Set to "auto" or "cpu" when MPS is unavailable.
IMSTE_MST_NEIGHBOR_BACKEND = "rinnd"


In [ ]:
import umap

umap_settings = {
    "n_components": 2,
    "n_neighbors": N_NEIGHBORS,
    "min_dist": 0.1,
    "repulsion_strength": 4,
    "metric": METRIC,
    "init": "recursive",
    "compatibility_layout": False,
    "nearest_neighbor_backend": "rinnd",
    "random_state": RANDOM_STATE,
    "n_epochs": UMAP_EPOCHS,
}
started = time.perf_counter()
Z_umap = umap.UMAP(verbose=True, **umap_settings).fit_transform(X_experiment)
umap_seconds = time.perf_counter() - started
print(f"Subset UMAP fit: {umap_seconds:.2f}s")


In [ ]:
from imste import IteratedMinimumSpanningTreeEmbedder
from imste.mst.famst import _ann_edges_rinnd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.manifold import SpectralEmbedding

# Independent local-graph spectral initialization.
edge_map = _ann_edges_rinnd(
    X_experiment, N_NEIGHBORS, RANDOM_STATE, verbose=False
)
pairs = np.asarray(list(edge_map), dtype=np.int64)
distances = np.fromiter(edge_map.values(), dtype=np.float64, count=len(edge_map))
rows = np.r_[pairs[:, 0], pairs[:, 1]]
cols = np.r_[pairs[:, 1], pairs[:, 0]]
distance_graph = coo_matrix(
    (np.r_[distances, distances], (rows, cols)),
    shape=(len(X_experiment), len(X_experiment)),
).tocsr()
local_scale = np.empty(len(X_experiment), dtype=np.float64)
for node in range(len(X_experiment)):
    incident = distance_graph.data[
        distance_graph.indptr[node] : distance_graph.indptr[node + 1]
    ]
    rank = min(N_NEIGHBORS, len(incident)) - 1
    local_scale[node] = np.partition(incident, rank)[rank]
affinity = np.exp(
    -(distances**2)
    / np.maximum(local_scale[pairs[:, 0]] * local_scale[pairs[:, 1]], 1e-12)
)
weights = np.r_[affinity, affinity]
neighborhood_graph = coo_matrix(
    (weights, (rows, cols)), shape=(len(X_experiment), len(X_experiment))
).tocsr()
components = connected_components(neighborhood_graph, directed=False, return_labels=False)
if components != 1:
    raise RuntimeError(f"Spectral initialization graph has {components} components.")
Z_spectral = SpectralEmbedding(
    n_components=2, affinity="precomputed", eigen_solver="lobpcg", eigen_tol=1e-3,
    random_state=RANDOM_STATE,
).fit_transform(neighborhood_graph)
Z_spectral /= max(float(np.max(np.abs(Z_spectral))), 1e-12)
print("Spectral initialization fit")

IMSTE_COMMON_SETTINGS = {
    "n_components": 2,
    "n_msts": IMSTE_N_MSTS,
    "n_epochs": IMSTE_N_EPOCHS,
    "batch_size": IMSTE_BATCH_SIZE,
    "learning_rate": IMSTE_LEARNING_RATE,
    "negative_ratio": IMSTE_NEGATIVE_RATIO,
    "lambda_rep": IMSTE_LAMBDA_REP,
    "random_state": RANDOM_STATE,
    "device": IMSTE_DEVICE,
    "logistic_margin": IMSTE_LOGISTIC_MARGIN,
    "logistic_temperature": IMSTE_LOGISTIC_TEMPERATURE,
    "mst_method": "famst",
    "mst_neighbor_backend": IMSTE_MST_NEIGHBOR_BACKEND,
    "mst_neighbors": N_NEIGHBORS,
    "mst_inter_component_edges": 5,
    "mst_representatives_per_component": 10,
    "mst_max_neighbors": None,
    "include_knn_attraction": False,
    "pca_init": True,
    "pca_init_scaling": "global",
    "multilevel_init": True,
    "multilevel_reduction_ratio": 8,
    "multilevel_base_size": 512,
    "multilevel_epochs": 10,
    "metric": METRIC,
    "verbose": False,
}

def fit_imste_variant(name, overrides=None, init_coordinates=None, init_id=None):
    settings = {**IMSTE_COMMON_SETTINGS, **(overrides or {})}
    estimator_settings = dict(settings)
    if init_coordinates is not None:
        estimator_settings["init_coordinates"] = init_coordinates
    started = time.perf_counter()
    estimator = IteratedMinimumSpanningTreeEmbedder(**estimator_settings)
    embedding = estimator.fit_transform(X_experiment)
    seconds = time.perf_counter() - started
    print(f"{name} fit: {seconds:.2f}s")
    return embedding, seconds

from imste.mst import build_mst_edges
from imste._estimator import _append_candidate_attraction_edges
from imste._estimator import _standardized_pca_initialization, _scale_initialization_coordinates
from imste._optimizer import optimize_embedding

def fit_weighted_knn_variant(strength):
    name = f"weighted_knn_strength_{strength:g}"
    started = time.perf_counter()
    candidate_batches = []
    mst_edges, mst_ranks = build_mst_edges(
        X_experiment,
        n_msts=IMSTE_N_MSTS,
        random_state=RANDOM_STATE,
        method="famst",
        neighbors=N_NEIGHBORS,
        inter_component_edges=5,
        max_neighbors=None,
        representatives_per_component=10,
        neighbor_device=IMSTE_DEVICE,
        neighbor_backend=IMSTE_MST_NEIGHBOR_BACKEND,
        return_ranks=True,
        metric=METRIC,
        candidate_edges_callback=candidate_batches.append,
    )
    candidate_edges = np.sort(np.asarray(candidate_batches[-1], dtype=np.int64), axis=1)
    n_samples = len(X_experiment)

    # Compute exact candidate-edge distances in bounded memory.
    candidate_distances = np.empty(len(candidate_edges), dtype=np.float64)
    for start in range(0, len(candidate_edges), 8192):
        stop = min(start + 8192, len(candidate_edges))
        delta = (
            np.asarray(X_experiment[candidate_edges[start:stop, 0]], dtype=np.float32)
            - np.asarray(X_experiment[candidate_edges[start:stop, 1]], dtype=np.float32)
        )
        candidate_distances[start:stop] = np.sqrt(np.einsum("ij,ij->i", delta, delta))

    rows = np.r_[candidate_edges[:, 0], candidate_edges[:, 1]]
    cols = np.r_[candidate_edges[:, 1], candidate_edges[:, 0]]
    distance_graph = coo_matrix(
        (np.r_[candidate_distances, candidate_distances], (rows, cols)),
        shape=(n_samples, n_samples),
    ).tocsr()
    local_rho = np.empty(n_samples, dtype=np.float64)
    local_sigma = np.empty(n_samples, dtype=np.float64)
    for node in range(n_samples):
        incident = distance_graph.data[
            distance_graph.indptr[node] : distance_graph.indptr[node + 1]
        ]
        ordered = np.partition(incident, min(N_NEIGHBORS, len(incident)) - 1)
        local_rho[node] = ordered[0]
        local_sigma[node] = max(ordered[min(N_NEIGHBORS, len(incident)) - 1] - ordered[0], 1e-8)
    left, right = candidate_edges.T
    left_membership = np.exp(-np.maximum(candidate_distances - local_rho[left], 0.0) / local_sigma[left])
    right_membership = np.exp(-np.maximum(candidate_distances - local_rho[right], 0.0) / local_sigma[right])
    candidate_weights = left_membership + right_membership - left_membership * right_membership

    mst_edges = np.sort(np.asarray(mst_edges, dtype=np.int64), axis=1)
    mst_keys = mst_edges[:, 0] * n_samples + mst_edges[:, 1]
    candidate_keys = candidate_edges[:, 0] * n_samples + candidate_edges[:, 1]
    unique_keys = np.union1d(mst_keys, candidate_keys)
    edge_weights = np.zeros(len(unique_keys), dtype=np.float64)
    np.add.at(edge_weights, np.searchsorted(unique_keys, mst_keys), 1.0 / mst_ranks)
    mst_weight_sum = float(np.sum(1.0 / mst_ranks))
    candidate_weights *= float(strength) * mst_weight_sum / max(float(candidate_weights.sum()), 1e-12)
    np.add.at(edge_weights, np.searchsorted(unique_keys, candidate_keys), candidate_weights)
    combined_edges = np.column_stack((unique_keys // n_samples, unique_keys % n_samples))

    coordinates = _standardized_pca_initialization(X_experiment, 2, RANDOM_STATE)
    coordinates = _scale_initialization_coordinates(coordinates, "global").astype(np.float32)
    seed_rng = np.random.RandomState(RANDOM_STATE)
    optimizer_seed = int(seed_rng.randint(0, np.iinfo(np.int32).max))
    kwargs = dict(
        n_epochs=IMSTE_N_EPOCHS,
        learning_rate=IMSTE_LEARNING_RATE,
        negative_ratio=IMSTE_NEGATIVE_RATIO,
        lambda_rep=IMSTE_LAMBDA_REP,
        epsilon=1e-4,
        margin=IMSTE_LOGISTIC_MARGIN,
        temperature=IMSTE_LOGISTIC_TEMPERATURE,
        random_state=optimizer_seed,
        schedule_by_rank=False,
        edge_weights=edge_weights,
    )
    if IMSTE_DEVICE == "mps":
        from imste._optimizer_mps import optimize_embedding_mps
        optimize_embedding_mps(
            coordinates, combined_edges, np.ones(len(combined_edges), dtype=np.int32),
            batch_size=IMSTE_BATCH_SIZE, **kwargs,
        )
    else:
        optimize_embedding(
            coordinates, combined_edges, np.ones(len(combined_edges), dtype=np.int32),
            **kwargs,
        )
    seconds = time.perf_counter() - started
    print(f"{name} fit: {seconds:.2f}s ({len(combined_edges):,} weighted edges)")
    return coordinates, seconds


def fit_neighbor_graph_multilevel_init():
    """Use local ANN edges only for coarsening/refinement, then optimize the MST."""
    name = "mst_objective_neighbor_graph_multilevel_init"
    started = time.perf_counter()
    candidate_batches = []
    graph_estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_COMMON_SETTINGS)
    options = graph_estimator._validated_fit_options()
    graph_estimator._activate_progress_reporting()
    mst_edges, mst_ranks = build_mst_edges(
        X_experiment,
        n_msts=IMSTE_N_MSTS,
        random_state=RANDOM_STATE,
        method="famst",
        neighbors=N_NEIGHBORS,
        inter_component_edges=5,
        max_neighbors=None,
        representatives_per_component=10,
        neighbor_device=IMSTE_DEVICE,
        neighbor_backend=IMSTE_MST_NEIGHBOR_BACKEND,
        return_ranks=True,
        metric=METRIC,
        candidate_edges_callback=candidate_batches.append,
    )
    init_edges, init_ranks = _append_candidate_attraction_edges(
        mst_edges, mst_ranks, candidate_batches[-1], len(X_experiment)
    )
    init_coordinates = graph_estimator._multilevel_initialization(
        len(X_experiment), 2, init_edges, init_ranks, options["resolved_device"],
        X_experiment, options,
    )
    if init_coordinates is None:
        raise RuntimeError("Neighbor-graph multilevel initialization produced no hierarchy.")

    estimator = IteratedMinimumSpanningTreeEmbedder(
        **IMSTE_COMMON_SETTINGS, init_coordinates=init_coordinates
    )
    embedding = estimator.fit_transform(X_experiment)
    seconds = time.perf_counter() - started
    print(f"{name} fit: {seconds:.2f}s")
    return embedding, seconds


def fit_edge_schedule_variants():
    """Hold graph, MPS optimizer, init, seed and loss fixed; change scheduling only."""
    prep_started = time.perf_counter()
    graph_estimator = IteratedMinimumSpanningTreeEmbedder(**IMSTE_COMMON_SETTINGS)
    options = graph_estimator._validated_fit_options()
    graph_estimator._activate_progress_reporting()
    edges, ranks = build_mst_edges(
        X_experiment,
        n_msts=IMSTE_N_MSTS,
        random_state=RANDOM_STATE,
        method="famst",
        neighbors=N_NEIGHBORS,
        inter_component_edges=5,
        max_neighbors=None,
        representatives_per_component=10,
        neighbor_device=IMSTE_DEVICE,
        neighbor_backend=IMSTE_MST_NEIGHBOR_BACKEND,
        return_ranks=True,
        metric=METRIC,
    )
    initialization = graph_estimator._multilevel_initialization(
        len(X_experiment), 2, edges, ranks, options["resolved_device"],
        X_experiment, options,
    )
    if initialization is None:
        raise RuntimeError("MST multilevel initialization produced no hierarchy.")
    shared_prep_seconds = time.perf_counter() - prep_started

    seed_rng = np.random.RandomState(RANDOM_STATE)
    optimizer_seed = int(seed_rng.randint(0, np.iinfo(np.int32).max))
    from imste._optimizer_mps import optimize_embedding_mps
    optimizer_kwargs = dict(
        n_epochs=IMSTE_N_EPOCHS,
        batch_size=IMSTE_BATCH_SIZE,
        learning_rate=IMSTE_LEARNING_RATE,
        negative_ratio=IMSTE_NEGATIVE_RATIO,
        lambda_rep=IMSTE_LAMBDA_REP,
        epsilon=1e-4,
        margin=IMSTE_LOGISTIC_MARGIN,
        temperature=IMSTE_LOGISTIC_TEMPERATURE,
        random_state=optimizer_seed,
    )
    rank_embedding = initialization.copy()
    started = time.perf_counter()
    optimize_embedding_mps(
        rank_embedding, edges, ranks, schedule_by_rank=True, **optimizer_kwargs
    )
    rank_total_seconds = shared_prep_seconds + time.perf_counter() - started

    weighted_embedding = initialization.copy()
    started = time.perf_counter()
    optimize_embedding_mps(
        weighted_embedding, edges, ranks,
        schedule_by_rank=False, edge_weights=1.0 / ranks, **optimizer_kwargs,
    )
    weighted_total_seconds = shared_prep_seconds + time.perf_counter() - started
    print(
        "Same MST graph/init: rank scheduling %.2fs; all-edge inverse-rank %.2fs"
        % (rank_total_seconds, weighted_total_seconds)
    )
    return rank_embedding, weighted_embedding, rank_total_seconds, weighted_total_seconds


In [ ]:
# Gravity is applied explicitly as a separate phase in these experiments.
IMSTE_COMMON_SETTINGS["gravity_mode"] = None


In [ ]:
# Each IMSTE variant starts independently; none uses UMAP coordinates.
# Change one optimizer/graph/initialization setting at a time.
Z_imste_baseline, imste_baseline_seconds = fit_imste_variant("pca_mst_baseline")
Z_imste_spectral, imste_spectral_seconds = fit_imste_variant(
    "pca_mst_spectral_init", init_coordinates=Z_spectral, init_id=spectral_key
)
Z_imste_pca_only, imste_pca_only_seconds = fit_imste_variant(
    "pca_only", {"multilevel_init": False}
)
Z_imste_500_epochs, imste_500_epochs_seconds = fit_imste_variant(
    "pca_mst_500_epochs", {"n_epochs": 500}
)
Z_imste_lr01, imste_lr01_seconds = fit_imste_variant(
    "pca_mst_lr001", {"learning_rate": 0.01}
)
Z_imste_repulsion_001, imste_repulsion_001_seconds = fit_imste_variant(
    "pca_mst_repulsion_001", {"lambda_rep": 0.01}
)
Z_imste_repulsion_01, imste_repulsion_01_seconds = fit_imste_variant(
    "pca_mst_repulsion_01", {"lambda_rep": 0.1}
)
Z_imste_repulsion_02, imste_repulsion_02_seconds = fit_imste_variant(
    "pca_mst_repulsion_02", {"lambda_rep": 0.2}
)
Z_imste_repulsion_03, imste_repulsion_03_seconds = fit_imste_variant(
    "pca_mst_repulsion_03", {"lambda_rep": 0.3}
)
Z_imste_negative5, imste_negative5_seconds = fit_imste_variant(
    "pca_mst_negative_ratio5", {"negative_ratio": 5}
)
Z_imste_margin1, imste_margin1_seconds = fit_imste_variant(
    "pca_mst_margin1", {"logistic_margin": 1.0}
)
Z_imste_wider_margin, imste_wider_margin_seconds = fit_imste_variant(
    "pca_mst_margin_05", {"logistic_margin": 0.5}
)
Z_imste_mst15, imste_mst15_seconds = fit_imste_variant(
    "pca_mst_15_trees", {"n_msts": 15}
)
Z_imste_neighbors30, imste_neighbors30_seconds = fit_imste_variant(
    "pca_mst_neighbors30", {"mst_neighbors": 30}
)
Z_imste_neighbors50, imste_neighbors50_seconds = fit_imste_variant(
    "pca_mst_neighbors50", {"mst_neighbors": 50}
)
Z_imste_knn, imste_knn_seconds = fit_imste_variant(
    "pca_mst_plus_candidate_neighbors", {"include_knn_attraction": True}
)
Z_weighted_knn_025, weighted_knn_025_seconds = fit_weighted_knn_variant(0.25)
Z_weighted_knn_1, weighted_knn_1_seconds = fit_weighted_knn_variant(1.0)
Z_weighted_knn_4, weighted_knn_4_seconds = fit_weighted_knn_variant(4.0)

Z_imste_knn_multilevel, imste_knn_multilevel_seconds = fit_neighbor_graph_multilevel_init()

(
    Z_imste_rank_scheduled,
    Z_imste_all_weighted,
    imste_rank_scheduled_seconds,
    imste_all_weighted_seconds,
) = fit_edge_schedule_variants()

Z_imste_random_multilevel, imste_random_multilevel_seconds = fit_imste_variant(
    "random_multilevel_init", {"pca_init": False, "multilevel_init": True}
)
Z_imste_random_direct, imste_random_direct_seconds = fit_imste_variant(
    "random_direct_init", {"pca_init": False, "multilevel_init": False}
)
Z_imste_pca_unscaled, imste_pca_unscaled_seconds = fit_imste_variant(
    "pca_unscaled_multilevel_init", {"pca_init_scaling": "none"}
)
Z_imste_pca_unscaled_direct, imste_pca_unscaled_direct_seconds = fit_imste_variant(
    "pca_unscaled_direct_init", {"pca_init_scaling": "none", "multilevel_init": False}
)


In [ ]:
# Diagnostic only: this tests how much the optimizer preserves a UMAP layout.
# It is not an independent IMSTE candidate and is disabled in the default sweep.
RUN_UMAP_WARMSTART_DIAGNOSTIC = False
if RUN_UMAP_WARMSTART_DIAGNOSTIC:
    umap_init_id = hashlib.sha256(
        np.ascontiguousarray(Z_umap, dtype=np.float32).view(np.uint8)
    ).hexdigest()[:16]
    Z_imste_umap_warmstart, imste_umap_warmstart_seconds = fit_imste_variant(
        "umap_warmstart_diagnostic_only",
        init_coordinates=Z_umap,
        init_id=f"umap-{umap_init_id}",
    )


In [ ]:
from sklearn.manifold import trustworthiness
from sklearn.neighbors import NearestNeighbors

METRIC_ANCHORS = min(2_000, len(X_experiment))
METRIC_K = min(15, max(1, len(X_experiment) // 2))
metric_rng = np.random.RandomState(2027)
anchor_indices = np.sort(metric_rng.choice(len(X_experiment), METRIC_ANCHORS, replace=False))
input_index = NearestNeighbors(n_neighbors=METRIC_K + 1, algorithm="brute", n_jobs=-1).fit(X_experiment)
input_neighbors = input_index.kneighbors(X_experiment[anchor_indices], return_distance=False)

def _without_self(neighbor_rows, query_indices, k):
    cleaned = []
    for row, query_index in zip(neighbor_rows, query_indices):
        cleaned.append([int(index) for index in row if index != query_index][:k])
    return cleaned

input_neighbors = _without_self(input_neighbors, anchor_indices, METRIC_K)
anchor_labels = np.asarray(labels_experiment)[anchor_indices]

def score_layout(name, coordinates):
    layout_index = NearestNeighbors(n_neighbors=METRIC_K + 1, algorithm="kd_tree", n_jobs=-1).fit(coordinates)
    distances, neighbor_rows = layout_index.kneighbors(coordinates[anchor_indices], return_distance=True)
    cleaned = _without_self(neighbor_rows, anchor_indices, METRIC_K)
    cleaned_distances = []
    for row, query_index in zip(neighbor_rows, anchor_indices):
        keep = row != query_index
        cleaned_distances.append(distances[len(cleaned_distances), keep][:METRIC_K])
    overlap = np.mean([
        len(set(source).intersection(target)) / METRIC_K
        for source, target in zip(input_neighbors, cleaned)
    ])
    neighbors = np.asarray(cleaned, dtype=np.int64)
    purity = float(np.mean(np.asarray(labels_experiment)[neighbors] == anchor_labels[:, None]))
    macro_purity = float(np.mean([
        np.mean(np.asarray(labels_experiment)[neighbors[anchor_labels == label]] == label)
        for label in np.unique(anchor_labels)
        if np.any(anchor_labels == label)
    ]))
    local_radius = float(np.median(np.asarray(cleaned_distances)[:, -1]))
    center = np.median(coordinates, axis=0)
    extent = float(np.median(np.linalg.norm(coordinates[anchor_indices] - center, axis=1)))
    with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
        trust = trustworthiness(
            X_experiment[trust_indices], coordinates[trust_indices], n_neighbors=METRIC_K
        )
    return {
        "layout": name,
        "trustworthiness": float(trust),
        "input_neighbor_overlap@15": float(overlap),
        "domain_purity@15": purity,
        "macro_domain_purity@15": macro_purity,
        "median_15NN_radius": local_radius,
        "median_extent": extent,
        "median_15NN_radius / median_extent": local_radius / max(extent, 1e-12),
    }

trust_indices = np.sort(metric_rng.choice(len(X_experiment), min(1_500, len(X_experiment)), replace=False))

score_rows = [
    score_layout("UMAP · reference", Z_umap),
    score_layout("IMSTE · PCA/multilevel", Z_imste_baseline),
    score_layout("IMSTE · rank-scheduled MST", Z_imste_rank_scheduled),
    score_layout("IMSTE · all MST edges, inverse-rank weights", Z_imste_all_weighted),
    score_layout("IMSTE · spectral init + MST", Z_imste_spectral),
    score_layout("IMSTE · ANN multilevel init, MST objective", Z_imste_knn_multilevel),
    score_layout("IMSTE · PCA only", Z_imste_pca_only),
    score_layout("IMSTE · random/multilevel init", Z_imste_random_multilevel),
    score_layout("IMSTE · random direct init", Z_imste_random_direct),
    score_layout("IMSTE · unscaled PCA/multilevel init", Z_imste_pca_unscaled),
    score_layout("IMSTE · unscaled direct PCA init", Z_imste_pca_unscaled_direct),
    score_layout("IMSTE · PCA/multilevel, 500 epochs", Z_imste_500_epochs),
    score_layout("IMSTE · lr=0.01", Z_imste_lr01),
    score_layout("IMSTE · repulsion=0.01", Z_imste_repulsion_001),
    score_layout("IMSTE · repulsion=0.10", Z_imste_repulsion_01),
    score_layout("IMSTE · repulsion=0.20", Z_imste_repulsion_02),
    score_layout("IMSTE · repulsion=0.30", Z_imste_repulsion_03),
    score_layout("IMSTE · negative ratio=5", Z_imste_negative5),
    score_layout("IMSTE · margin=1.0", Z_imste_margin1),
    score_layout("IMSTE · margin=0.5", Z_imste_wider_margin),
    score_layout("IMSTE · 15 MSTs", Z_imste_mst15),
    score_layout("IMSTE · 30 candidate neighbors", Z_imste_neighbors30),
    score_layout("IMSTE · 50 candidate neighbors", Z_imste_neighbors50),
    score_layout("IMSTE · MST + neighbor attractions", Z_imste_knn),
    score_layout("IMSTE · weighted ANN, 0.25× MST weight", Z_weighted_knn_025),
    score_layout("IMSTE · weighted ANN, 1× MST weight", Z_weighted_knn_1),
    score_layout("IMSTE · weighted ANN, 4× MST weight", Z_weighted_knn_4),
]
if RUN_UMAP_WARMSTART_DIAGNOSTIC:
    score_rows.append(score_layout("IMSTE · UMAP-start diagnostic only", Z_imste_umap_warmstart))

scores = pd.DataFrame(score_rows)
display(scores)


## Current independent IMSTE results

On the deterministic balanced 32k subset, the UMAP reference scored 0.926 trustworthiness, 0.212 input 15-NN overlap, 0.710 domain purity, and 0.0109 normalized 15-NN radius. The standard independent IMSTE baseline scored 0.825, 0.054, 0.603, and 0.0164 respectively.

The unscaled-PCA/multilevel run is the first promising compactness candidate: its normalized radius was 0.00895 versus UMAP's 0.01086, with trustworthiness 0.874. Its input 15-NN overlap (0.099) and domain purity (0.679) still lag UMAP (0.212 and 0.710), so the 32k result alone does not meet the fidelity target. A direct unscaled-PCA run is included next to separate coordinate scaling from multilevel refinement. No tested IMSTE configuration has yet met both UMAP's local fidelity and compactness. Raising `lambda_rep` to 0.3 gave the best IMSTE fidelity in this sweep (0.844 trustworthiness, 0.064 overlap, 0.633 purity), but its normalized 15-NN radius grew to 0.0235. Raising the logistic margin to 1.0 produced only small fidelity gains and a normalized radius of 0.0169. Increasing negative samples from 1 to 5 had negligible effect. Raising epochs to 500 improved trustworthiness to 0.831 but increased normalized radius to 0.0180. Increasing candidate neighbors to 30 or 50 had little effect. Spectral initialization scored 0.831 trustworthiness but had a 0.0218 normalized radius. Using ANN edges to construct the multilevel start slightly reduced normalized radius to 0.0170, but trustworthiness and domain purity fell. Weighted ANN attractions took about 5.2s end-to-end versus 2.9s for the baseline and did not improve the overall tradeoff. Raw median 15-NN radius and median radial extent are also reported because embedding coordinate scales are arbitrary; compare compactness using the normalized radius.

Replacing rank-staggered updates with all-edge updates weighted by inverse rank was also a tie: trustworthiness changed from 0.8247 to 0.8250, overlap from 0.0538 to 0.0537, purity from 0.6031 to 0.6035, and normalized radius from 0.01635 to 0.01652; total time rose from 2.79s to 3.62s. These controlled results rule out simple repulsion, epoch-count, neighbor-count, initialization, and scheduling changes as sufficient fixes. **The fan-shaped independent IMSTE layout is still unresolved.** The next useful work is a change to the graph optimization objective or edge scheduling, followed by confirmation on the full 640k subset; these 32k diagnostics are for screening only. No UMAP coordinates are used to initialize an IMSTE run.

## 640k scale confirmation

A deterministic random 640,000-row subset of the current 1.2m-row source cache (`dataset_id=4270d8e4b4e3ce22990a`, subset ID `3605bdd0d9fb967ec06f`) was used to check whether the 32k balanced-sample findings transfer. IMSTE variants used the same 5-tree FAMST graph, RINND search, MPS optimizer, 100 epochs, learning rate 0.05, batch size 32,768, and seed 42. Each run initialized independently from its own PCA settings; no run used UMAP coordinates. Scores use exact input distances for 512 fixed anchors against all 640k rows, with 15-neighbor overlap, trustworthiness, and domain purity.

| Layout | Fit (s) | Trustworthiness | Input 15-NN overlap | Domain purity | Relative 15-NN radius |
|---|---:|---:|---:|---:|---:|
| UMAP reference | 309 | 0.977 | 0.0772 | 0.843 | 0.00354 |
| IMSTE global PCA + multilevel | 87.6 | 0.843 | 0.0107 | 0.767 | 0.00202 |
| IMSTE unscaled PCA + multilevel | 89.5 | 0.860 | 0.0133 | 0.795 | 0.00182 |
| IMSTE direct PCA | 64.3 | 0.843 | 0.0100 | 0.789 | 0.00371 |
| IMSTE MST + local-neighbor attractions | 237.5 | 0.870 | 0.0095 | 0.797 | 0.00267 |
| IMSTE, `lambda_rep=0.01` | 72.0 | 0.852 | 0.0121 | 0.773 | 0.00110 |
| IMSTE, `lambda_rep=0.025` | 82.2 | 0.856 | 0.0118 | 0.785 | 0.00197 |

The unscaled-PCA run slightly improves the measured metrics but remains a long fan in the fixed render. The local-neighbor attraction run is also fan-shaped, takes 2.7× the baseline time, and has lower input-neighbor overlap. `lambda_rep=0.01` collapses into a narrow line; `0.025` remains fan-shaped and has almost the same relative local radius as the baseline. Direct PCA is more diffuse. **None of these settings meets the compact, interpretable layout goal**, so the standard IMSTE configuration remains the reference while further work focuses on the graph objective and initialization hierarchy.

The UMAP reference used installed UMAP 0.5.12 with its native spectral initialization and default approximate-neighbor implementation, 15 neighbors, `min_dist=0.1`, repulsion 4, and 200 epochs. Its runtime is a reference measurement, not an apples-to-apples backend comparison with the older RINND-enabled notebook run. The full-scale run artifacts and renders are cached locally under `data/arxiv/cache/balanced_experiments/confirm640_*`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex
from IPython.display import display
import importlib
import imste.visualization as _imste_visualization
_imste_visualization = importlib.reload(_imste_visualization)
render_datashader_embedding = _imste_visualization.render_datashader_embedding

def render_embedding_preserving_aspect(embedding, labels=None, **kwargs):
    """Render without geometric stretching; support older IMSTE installs too."""
    import inspect
    options = dict(kwargs)
    renderer_parameters = inspect.signature(render_datashader_embedding).parameters
    if "preserve_aspect" in renderer_parameters:
        options["preserve_aspect"] = True
    else:
        coordinates = np.asarray(embedding)
        width = int(options.get("width", 1000))
        height = int(options.get("height", 1000))
        x_limits = np.asarray(
            options.get("x_range") or (coordinates[:, 0].min(), coordinates[:, 0].max()),
            dtype=np.float64,
        )
        y_limits = np.asarray(
            options.get("y_range") or (coordinates[:, 1].min(), coordinates[:, 1].max()),
            dtype=np.float64,
        )
        pixel_scale = max(
            (x_limits[1] - x_limits[0]) / width,
            (y_limits[1] - y_limits[0]) / height,
            np.finfo(np.float64).eps,
        )
        x_center, y_center = float(x_limits.mean()), float(y_limits.mean())
        options["x_range"] = (
            x_center - pixel_scale * width / 2,
            x_center + pixel_scale * width / 2,
        )
        options["y_range"] = (
            y_center - pixel_scale * height / 2,
            y_center + pixel_scale * height / 2,
        )
    return render_datashader_embedding(embedding, labels, **options)


palette = plt.get_cmap("tab20")
class_order = list(labels_experiment.categories)
color_key = {label: to_hex(palette(i % palette.N)) for i, label in enumerate(class_order)}
label_values = pd.Series(np.asarray(labels_experiment)).astype(str)
layouts = [
    ("UMAP reference", Z_umap, umap_seconds),
    ("IMSTE PCA/multilevel", Z_imste_baseline, imste_baseline_seconds),
    ("IMSTE unscaled PCA/multilevel", Z_imste_pca_unscaled, imste_pca_unscaled_seconds),
    ("IMSTE unscaled direct PCA", Z_imste_pca_unscaled_direct, imste_pca_unscaled_direct_seconds),
    ("IMSTE rank-scheduled", Z_imste_rank_scheduled, imste_rank_scheduled_seconds),
    ("IMSTE all edges, inv-rank", Z_imste_all_weighted, imste_all_weighted_seconds),
    ("IMSTE repulsion=0.30", Z_imste_repulsion_03, imste_repulsion_03_seconds),
    ("IMSTE 500 epochs", Z_imste_500_epochs, imste_500_epochs_seconds),
    ("IMSTE ANN multilevel init", Z_imste_knn_multilevel, imste_knn_multilevel_seconds),
    ("IMSTE weighted ANN 0.25×", Z_weighted_knn_025, weighted_knn_025_seconds),
]
fig, axes = plt.subplots(2, 5, figsize=(25, 10), constrained_layout=True)
for ax, (name, coordinates, seconds) in zip(axes.ravel(), layouts):
    image = render_embedding_preserving_aspect(
        coordinates[:, :2], label_values, width=900, height=750,
        color_key=color_key, min_alpha=40, background="white", spread_px=1,
    )
    ax.imshow(image.to_pil())
    ax.set_title(f"{name} · {len(X_experiment):,} rows · {seconds:.1f}s")
    ax.set_axis_off()

display(fig)
plt.close(fig)


## Density-weighted attraction phase (prototype)

This first second-phase prototype starts from the unscaled PCA + multilevel IMSTE embedding and moves each point toward a density-weighted centroid of its RINND neighbors. Labels are used only for coloring and post-fit metrics, including an 8-domain silhouette score. They do not enter initialization or force calculations.

This early sweep used 16 candidate neighbors (it predates matching FAMST's exact 15-neighbor graph settings). It behaved like diffusion and did not tighten neighborhoods reliably. Treat it as a rejected local-smoothing prototype; the dense-core experiment below uses FAMST's RINND settings.

In [ ]:
# Density-weighted second phase, using only X_experiment and the preferred IMSTE layout.
# UMAP coordinates are not involved in this experiment.
from rinnd import RINND
from sklearn.metrics import silhouette_score

GRAVITY_NEIGHBORS = 15
GRAVITY_MASS_EXPONENT = 2.0
_gravity_index = RINND(
    np.ascontiguousarray(X_experiment, dtype=np.float32),
    metric="euclidean", n_neighbors=GRAVITY_NEIGHBORS + 1,
    random_state=RANDOM_STATE, n_trees=8, n_iters=15,
    max_candidates=30, n_jobs=-1, verbose=False,
)
_gravity_indices, _gravity_distances = _gravity_index.neighbor_graph
_gravity_rows, _gravity_cols, _gravity_dists = [], [], []
for _i in range(len(X_experiment)):
    _keep = _gravity_indices[_i] != _i
    _js = np.asarray(_gravity_indices[_i][_keep][:GRAVITY_NEIGHBORS], dtype=np.int64)
    _ds = np.asarray(_gravity_distances[_i][_keep][:GRAVITY_NEIGHBORS], dtype=np.float32)
    _gravity_rows.extend([_i] * len(_js)); _gravity_cols.extend(_js.tolist()); _gravity_dists.extend(_ds.tolist())
_gravity_rows = np.asarray(_gravity_rows, dtype=np.int64)
_gravity_cols = np.asarray(_gravity_cols, dtype=np.int64)
_gravity_dists = np.asarray(_gravity_dists, dtype=np.float32)
_gravity_kth = np.asarray([
    np.max(_gravity_dists[_gravity_rows == _i]) for _i in range(len(X_experiment))
], dtype=np.float32)
_log_density = -np.log(np.maximum(_gravity_kth, 1e-12))
_log_density = (_log_density - np.median(_log_density)) / (
    np.median(np.abs(_log_density - np.median(_log_density))) + 1e-12
)
_gravity_mass = np.exp(np.clip(_log_density, -3, 3))
_gravity_base_weights = np.exp(
    -_gravity_dists / np.maximum(_gravity_kth[_gravity_rows], 1e-12)
)

def density_gravity_phase(coordinates, strength, steps, mass_exponent=GRAVITY_MASS_EXPONENT):
    edge_weights = _gravity_base_weights * _gravity_mass[_gravity_cols] ** mass_exponent
    row_weight = np.bincount(_gravity_rows, weights=edge_weights, minlength=len(coordinates))
    result = np.asarray(coordinates, dtype=np.float64).copy()
    for _ in range(steps):
        neighbor_mass = np.zeros_like(result)
        np.add.at(neighbor_mass, _gravity_rows, edge_weights[:, None] * result[_gravity_cols])
        neighbor_mass /= np.maximum(row_weight[:, None], 1e-15)
        result += strength * (neighbor_mass - result)
        result -= result.mean(axis=0)
    return result.astype(np.float32)

# Small sweep: keep the base embedding fixed and vary only force strength/steps.
gravity_configs = [
    (2.0, 0.03, 1), (2.0, 0.08, 4), (2.0, 0.08, 8),
    (2.0, 0.15, 4), (2.0, 0.15, 8), (2.0, 0.25, 8),
    (4.0, 0.25, 8),
]
gravity_results = [{"configuration": "baseline", **score_layout("baseline", Z_imste_pca_unscaled),
                   "label_silhouette": silhouette_score(
                       np.asarray(Z_imste_pca_unscaled), np.asarray(labels_experiment),
                       sample_size=min(5_000, len(X_experiment)), random_state=2027
                   )}]
gravity_embeddings = {"baseline": Z_imste_pca_unscaled}
for _mass, _strength, _steps in gravity_configs:
    _name = f"mass={_mass:g}, strength={_strength:g}, steps={_steps}"
    _z = density_gravity_phase(Z_imste_pca_unscaled, _strength, _steps, _mass)
    gravity_embeddings[_name] = _z
    gravity_results.append({"configuration": _name, **score_layout(_name, _z),
                           "label_silhouette": silhouette_score(
                               np.asarray(_z), np.asarray(labels_experiment),
                               sample_size=min(5_000, len(X_experiment)), random_state=2027
                           )})
gravity_table = pd.DataFrame(gravity_results)[[
    "configuration", "trustworthiness", "input_neighbor_overlap@15",
    "domain_purity@15", "label_silhouette",
    "median_15NN_radius / median_extent"
]]
display(gravity_table.round(4))

# Render the baseline and four intermediate force settings side by side.
from imste.visualization import render_datashader_embedding
from matplotlib.colors import to_hex
_gravity_palette = plt.get_cmap("tab20")
_gravity_labels = np.asarray(labels_experiment).astype(str)
_gravity_color_key = {label: to_hex(_gravity_palette(i % _gravity_palette.N))
                      for i, label in enumerate(pd.unique(_gravity_labels))}
_gravity_plot_names = ["baseline"] + [
    "mass=2, strength=0.08, steps=4", "mass=2, strength=0.08, steps=8",
    "mass=2, strength=0.15, steps=4", "mass=2, strength=0.15, steps=8",
]
fig, axes = plt.subplots(1, 5, figsize=(22, 5), constrained_layout=True)
for _ax, _name in zip(axes, _gravity_plot_names):
    _image = render_embedding_preserving_aspect(
        gravity_embeddings[_name], _gravity_labels, width=700, height=600,
        color_key=_gravity_color_key, min_alpha=40, background="white", spread_px=1,
    )
    _ax.imshow(_image.to_pil()); _ax.set_title(_name, fontsize=10); _ax.set_axis_off()
display(fig); plt.close(fig)


## Graph-region mass attraction: dense-core version

The local neighbor-centroid pass behaved like diffusion. This version partitions the feature-space RINND graph into Leiden communities, estimates density from each point's 15th-neighbor radius, and computes each community center from its densest **2%** of points. Every point in a community is pulled toward that dense core. The force coefficient is `strength × (region_mass / median_region_mass)`, capped at 0.9; region mass is the sum of density weights.

RINND uses the same graph-search settings as FAMST here: 15 neighbors, 8 trees, `max_candidates=22`, and `n_iters=max(5, round(log2(n)))`. Domain labels are used only for plot colors and evaluation. The 32k sweep keeps the unscaled PCA/multilevel IMSTE start fixed.

In [ ]:
# Optional research dependencies for graph community detection.
try:
    import igraph as ig
    import leidenalg
except ImportError as exc:
    raise ImportError("Install experiment dependencies with `python -m pip install igraph leidenalg`.") from exc
from rinnd import RINND
from sklearn.metrics import silhouette_score

REGION_NEIGHBORS = 15
REGION_RESOLUTION = 1.2
REGION_CORE_QUANTILE = 0.98
_region_index = RINND(
    np.ascontiguousarray(X_experiment, dtype=np.float32), metric="euclidean",
    n_neighbors=REGION_NEIGHBORS, random_state=RANDOM_STATE,
    n_trees=8, n_iters=max(5, int(round(np.log2(len(X_experiment))))),
    max_candidates=max(20, min(60, int(1.5 * REGION_NEIGHBORS))), n_jobs=-1, verbose=False,
)
_region_indices, _region_distances = _region_index.neighbor_graph
_region_rows = np.repeat(np.arange(len(X_experiment)), _region_indices.shape[1])
_region_cols = np.asarray(_region_indices).reshape(-1).astype(np.int64)
_region_dists = np.asarray(_region_distances).reshape(-1).astype(np.float32)
_region_valid = (_region_cols != _region_rows) & np.isfinite(_region_dists)
_region_rows, _region_cols, _region_dists = (
    _region_rows[_region_valid], _region_cols[_region_valid], _region_dists[_region_valid]
)
_region_radius = np.max(
    np.where(_region_indices != np.arange(len(X_experiment))[:, None],
             _region_distances, -np.inf), axis=1
).astype(np.float32)
_node_mass_raw = (np.median(_region_radius) / np.maximum(_region_radius, 1e-12)) ** 2
_region_node_mass = np.clip(
    _node_mass_raw, np.percentile(_node_mass_raw, 1), np.percentile(_node_mass_raw, 99)
)
_region_edge_weights = np.exp(
    -_region_dists / np.maximum(_region_radius[_region_rows], 1e-12)
).astype(np.float32)
_region_graph = ig.Graph(
    n=len(X_experiment), edges=zip(_region_rows.tolist(), _region_cols.tolist()), directed=False
)
_region_graph.es["weight"] = _region_edge_weights.tolist()
_region_graph.simplify(multiple=True, loops=True, combine_edges={"weight": "sum"})
_region_partition = leidenalg.find_partition(
    _region_graph, leidenalg.RBConfigurationVertexPartition,
    weights="weight", resolution_parameter=REGION_RESOLUTION, seed=RANDOM_STATE,
)
_region_membership = np.asarray(_region_partition.membership, dtype=np.int64)
_region_count = int(_region_membership.max()) + 1
_region_masses = np.bincount(
    _region_membership, weights=_region_node_mass, minlength=_region_count
)
_region_thresholds = np.asarray([
    np.quantile(_region_node_mass[_region_membership == _j], REGION_CORE_QUANTILE)
    for _j in range(_region_count)
])
_region_core = _region_node_mass >= _region_thresholds[_region_membership]
_region_core_masses = np.bincount(
    _region_membership[_region_core], weights=_region_node_mass[_region_core], minlength=_region_count
)
_region_centers = np.zeros((_region_count, Z_imste_pca_unscaled.shape[1]), dtype=np.float64)
np.add.at(
    _region_centers, _region_membership[_region_core],
    np.asarray(Z_imste_pca_unscaled, dtype=np.float64)[_region_core]
    * _region_node_mass[_region_core, None],
)
_region_centers /= _region_core_masses[:, None]
_region_median_mass = np.median(_region_masses)

def apply_region_mass_attraction(coordinates, strength):
    region_strength = np.clip(
        strength * (_region_masses / _region_median_mass), 0.0, 0.9
    )
    result = np.asarray(coordinates, dtype=np.float64).copy()
    result += region_strength[_region_membership, None] * (
        _region_centers[_region_membership] - result
    )
    result -= result.mean(axis=0)
    return result.astype(np.float32)

region_strengths = [0.2, 0.4, 0.7]
region_results = [{
    "configuration": "baseline", **score_layout("baseline", Z_imste_pca_unscaled),
    "label_silhouette": silhouette_score(
        np.asarray(Z_imste_pca_unscaled), np.asarray(labels_experiment),
        sample_size=min(5_000, len(X_experiment)), random_state=2027
    ),
}]
region_embeddings = {"baseline": Z_imste_pca_unscaled}
for _strength in region_strengths:
    _name = f"strength={_strength:g}, mass_power=1, core=98%"
    _z = apply_region_mass_attraction(Z_imste_pca_unscaled, _strength)
    region_embeddings[_name] = _z
    region_results.append({
        "configuration": _name, **score_layout(_name, _z),
        "label_silhouette": silhouette_score(
            np.asarray(_z), np.asarray(labels_experiment),
            sample_size=min(5_000, len(X_experiment)), random_state=2027
        ),
    })
region_table = pd.DataFrame(region_results)[[
    "configuration", "trustworthiness", "input_neighbor_overlap@15",
    "domain_purity@15", "label_silhouette",
    "median_15NN_radius / median_extent",
]]
display(region_table.round(4))

_region_labels = np.asarray(labels_experiment).astype(str)
_region_palette = plt.get_cmap("tab20")
_region_color_key = {label: to_hex(_region_palette(i % _region_palette.N))
                      for i, label in enumerate(pd.unique(_region_labels))}
_region_names = ["baseline"] + [
    f"strength={s:g}, mass_power=1, core=98%" for s in region_strengths
]
fig, axes = plt.subplots(1, 4, figsize=(21, 5), constrained_layout=True)
for _ax, _name in zip(axes, _region_names):
    _image = render_embedding_preserving_aspect(
        region_embeddings[_name], _region_labels, width=700, height=650,
        color_key=_region_color_key, min_alpha=40, background="white", spread_px=1,
    )
    _ax.imshow(_image.to_pil()); _ax.set_title(_name, fontsize=9); _ax.set_axis_off()
display(fig); plt.close(fig)


## Full-size confirmation: 640,000 papers

All rows use the same deterministic random 640k sample (`data_id=3605bdd0d9fb967ec06f`, seed 42). Trustworthiness and input-neighbor overlap use exact 15-neighbor ranks for 512 fixed anchors; purity uses those anchors; silhouette is sampled over 5,000 rows with seed 2027. Labels are used only for post-fit scores and plot colors.

The current compactness candidate is **3 MSTs plus density-core gravity on a 21-region Leiden partition (resolution 0.8, core quantile 0.95, strength 0.4)**. Compared with the unscaled-PCA 5-MST baseline, trustworthiness changes from 0.860 to 0.867, input 15-NN overlap from 0.0133 to 0.0273, domain purity from 0.795 to 0.806, macro purity from 0.400 to 0.439, and silhouette from 0.054 to 0.145. The normalized 15-NN radius falls from 0.00182 to 0.00075 (59% smaller). The 3-MST fit takes about 50s; RINND graph construction and Leiden partition take about 108s, for about 157s end-to-end, compared with 309s for the UMAP reference.

Resolution 3 with the same strength has higher trustworthiness (0.917) but a larger normalized local radius (0.00104). The fixed rendering preserves the coordinate aspect ratio. It shows resolution 0.8 gives more compact, better separated regions while retaining a U-shaped global fan. This is the best tested compactness tradeoff, not a complete solution to the global shape. One MST scores higher but produces sparse, strand-like branches; adding local attraction edges did not improve the overall tradeoff. 

A cross-region mass-force follow-up moved the 21 Leiden-region centers with softened pairwise attraction and short-range repulsion, leaving each region's internal coordinates rigid. Across four settings (20 steps; equilibrium spacing 0.5–1.0× median center spacing; step 0.05–0.1), trustworthiness fell to 0.849–0.862, input 15-NN overlap to 0.0243–0.0267, and label silhouette to 0.057–0.122. This is rejected: moving whole centers disrupts semantic neighborhoods even when macro domain purity rises in one setting. UMAP remains a reference only; no UMAP coordinates initialize any IMSTE experiment.


### Diagnostic: UMAP warm start (not an IMSTE candidate)

To distinguish the optimizer from the initialization hierarchy, a separate run started from the previously computed 640k UMAP coordinates and used the same 3-MST FAMST graph, RINND search, MPS optimizer, loss settings, and 100 epochs as the independent 3-MST baseline. Explicit coordinates bypass multilevel initialization, so this is a diagnostic comparison rather than a deployable IMSTE configuration. It fit in 36.7s, versus 49.6s for the independent baseline and 308.8s for UMAP.

The fixed render recovers UMAP-like separated regions from the warm start. On the same 512 anchors, warm-start IMSTE had 0.901 anchor-subset trustworthiness, 0.0615 exact full-data input 15-NN overlap, 0.835 domain purity, and 0.528 macro domain purity. The independent baseline had 0.852, 0.0195, 0.800, and 0.395; UMAP had 0.901, 0.0772, 0.843, and 0.563. The warm-start local-radius ratio was 0.00328, compared with 0.00168 for the independent baseline and 0.00360 for UMAP. These radius ratios do not alone measure semantic region compactness: the fan layout has smaller local radii but visibly lacks separated regions. The fixed rendering and neighborhood purity show that the optimizer largely retains the starting geometry. This points to the initialization/graph-layout path as a major source of the visual gap; the diagnostic itself is excluded from the independent IMSTE selection.


### Stronger density-core pull at the same 21-region resolution

With strength increased from 0.4 to 0.7, the same RINND/Leiden graph contracts the layout further without using labels in the force. On the same 512 anchors, trustworthiness changed from 0.852 to 0.867, exact input 15-NN overlap from 0.0195 to 0.0313, domain purity from 0.800 to 0.822, and macro purity from 0.395 to 0.481. Domain silhouette decreased from 0.146 to 0.136. The normalized local radius fell from 0.00168 to 0.000322. Cold end-to-end time is about 161s (50s base fit + 111s graph/partition/phase), versus 309s for UMAP. The fixed render still has the broad fan, so this is a strong local-compaction candidate but does not satisfy the global-shape goal. Raising core pull does not solve the remaining problem. A quick 0.9 setting increased anchor trustworthiness to 0.871, exact overlap to 0.0322, domain purity to 0.827, and macro purity to 0.491, while reducing the local-radius ratio to 0.000273; domain silhouette fell again to 0.128. Its fixed render is more fragmented, with visible detached specks, and still follows the fan. We do not select it.

In [ ]:
# Captured fixed-seed 640k comparisons. Labels appear only in scoring and color.
full_size_results = pd.DataFrame([
    {'layout': 'UMAP reference (no IMSTE initialization)', 'total seconds': 308.84, 'trustworthiness': 0.977022, 'input 15-NN overlap': 0.077214, 'domain purity': 0.842969, 'macro domain purity': 0.563316, 'label silhouette (5k)': 0.07472, 'relative local radius': 0.003539},
    {'layout': 'IMSTE 5 MST baseline', 'total seconds': 89.55, 'trustworthiness': 0.860163, 'input 15-NN overlap': 0.013281, 'domain purity': 0.795052, 'macro domain purity': 0.399725, 'label silhouette (5k)': 0.054447, 'relative local radius': 0.001816},
    {'layout': 'IMSTE 1 MST (strand-like; reject)', 'total seconds': 29.87, 'trustworthiness': 0.962981, 'input 15-NN overlap': 0.147396, 'domain purity': 0.817708, 'macro domain purity': 0.540108, 'label silhouette (5k)': 0.183078, 'relative local radius': 0.000102},
    {'layout': 'IMSTE 2 MST', 'total seconds': 38.92, 'trustworthiness': 0.855865, 'input 15-NN overlap': 0.019922, 'domain purity': 0.779036, 'macro domain purity': 0.3774, 'label silhouette (5k)': 0.081952, 'relative local radius': 0.001427},
    {'layout': 'IMSTE 3 MST (leading base)', 'total seconds': 49.64, 'trustworthiness': 0.87375, 'input 15-NN overlap': 0.019531, 'domain purity': 0.79987, 'macro domain purity': 0.394995, 'label silhouette (5k)': 0.145551, 'relative local radius': 0.001704},
    {'layout': '3 MST, repulsion 0.025', 'total seconds': 94.78, 'trustworthiness': 0.870589, 'input 15-NN overlap': 0.01888, 'domain purity': 0.801823, 'macro domain purity': 0.389218, 'label silhouette (5k)': 0.145091, 'relative local radius': 0.001561},
    {'layout': '3 MST, repulsion 0.075', 'total seconds': 67.28, 'trustworthiness': 0.87529, 'input 15-NN overlap': 0.020443, 'domain purity': 0.801042, 'macro domain purity': 0.39323, 'label silhouette (5k)': 0.146059, 'relative local radius': 0.001879},
    {'layout': '5 MST + sparse local edges (weight 0.2)', 'total seconds': 114.3, 'trustworthiness': 0.860793, 'input 15-NN overlap': 0.016797, 'domain purity': 0.790625, 'macro domain purity': 0.389516, 'label silhouette (5k)': 0.054864, 'relative local radius': 0.001873},
    {'layout': '3 MST + 90 regions, gravity 0.2', 'total seconds': 156.8, 'trustworthiness': 0.887441, 'input 15-NN overlap': 0.024479, 'domain purity': 0.807031, 'macro domain purity': 0.395973, 'label silhouette (5k)': 0.139088, 'relative local radius': 0.001432},
    {'layout': '3 MST + 90 regions, gravity 0.3', 'total seconds': 156.8, 'trustworthiness': 0.894215, 'input 15-NN overlap': 0.028125, 'domain purity': 0.807552, 'macro domain purity': 0.401121, 'label silhouette (5k)': 0.134999, 'relative local radius': 0.00129},
    {'layout': '3 MST + 90 regions, gravity 0.4 (leading candidate)', 'total seconds': 156.8, 'trustworthiness': 0.916896, 'input 15-NN overlap': 0.033724, 'domain purity': 0.81849, 'macro domain purity': 0.438073, 'label silhouette (5k)': 0.13106, 'relative local radius': 0.001041},
    {'layout': '3 MST + 90 regions, gravity 0.7', 'total seconds': 156.8, 'trustworthiness': 0.935638, 'input 15-NN overlap': 0.040755, 'domain purity': 0.830208, 'macro domain purity': 0.452147, 'label silhouette (5k)': 0.117609, 'relative local radius': 0.000505},
    {'layout': '3 MST + 21 regions, gravity 0.4 (compactness candidate)', 'total seconds': 157.37, 'trustworthiness': 0.867469, 'input 15-NN overlap': 0.027344, 'domain purity': 0.806250, 'macro domain purity': 0.438620, 'label silhouette (5k)': 0.145280, 'relative local radius': 0.000750},
])
display(full_size_results.round(4))
print("Saved plot assets were removed; rerun the experiment cells to regenerate layouts and figures.")


## Independent quotient-layout candidate on the 640k sample

The full-size screen points to initialization as the main source of the global-shape gap. Starting IMSTE from a UMAP layout remains diagnostic-only. As an independent alternative, I built a RINND/Leiden graph, laid out its community quotient with Fruchterman–Reingold, added small per-community SVD offsets, and then ran IMSTE followed by density-core gravity. Labels were used only for the reported scores and colors.

At 32k, the quotient initialization plus gravity improved trustworthiness, exact input 15-NN overlap, and domain purity versus PCA/multilevel IMSTE. The render changes the fan into compact regions, though sparse points still bridge the regions. Replacing the force-directed start with an igraph DRL layout did not help: the result remained fan-shaped, and its 55s layout setup was expensive. The DRL render is shown for that rejected trial.



The 640k candidate uses the same three-MST optimizer as the baseline: 100 epochs, batch size 32,768, MPS, FAMST with RINND 15-neighbor search, `lambda_rep=0.05`, and no multilevel pass. Its independent start uses 21 Leiden regions at resolution 0.8, 210 quotient links, and within-region SVD offsets at 0.15 times the median quotient-center spacing. Density-core gravity uses strength 0.7, core quantile 0.95, and mass power 1.0.

| 640k layout | Fit time | Trustworthiness (512-row diagnostic subset) | Exact input 15-NN overlap | Domain purity | Macro domain purity | Label silhouette (5k) | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| 3-MST IMSTE baseline | 49.6 s | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Quotient FR + IMSTE + gravity 0.7 | 50.8 s warm; about 158.5 s including the one-time Leiden graph build | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

All three rows use the same deterministic 640k subset and 512 anchors. The 15-NN overlap is exact against all 640k input rows; trustworthiness is evaluated within the fixed 512-row diagnostic subset. The candidate improves every reported metric versus independent IMSTE, cuts the normalized local radius by 54%, and remains about half the UMAP runtime even including the one-time graph build. A gravity-only sweep found strength 0.9 slightly improves local radius, trustworthiness, and overlap, but lowers silhouette; 0.5 has the highest silhouette but weaker compactness. Strength 0.7 is the better visual balance in the fixed renders.


The resolution screen also favors 0.8 over 3.0. At 32k, resolution 3 produced 53 communities and still improved trustworthiness (0.899) and overlap (0.107), but the normalized radius was larger (0.00578 versus 0.00392), silhouette was lower (0.098 versus 0.114), and the render had more isolated fragments.


The result is a substantial compactness improvement, but the global fan has not disappeared completely: dust and bridges still connect some regions. A 640k `lambda_rep=0.3` run kept essentially the same global shape while increasing the normalized local radius from 0.00168 to 0.00322, so stronger repulsion is not the remedy. The current best independent configuration is quotient FR initialization + three-MST IMSTE + density-core gravity at strength 0.7; treat it as the leading candidate, not a completed resolution of the remaining global-shape gap.


## Single-phase quotient-layout compactness screen

I compared raw quotient-edge weights with square-root weights in the Fruchterman–Reingold community layout on the same 640,000-row sample. Both candidates use the same 21 Leiden regions, local SVD offsets, three-MST IMSTE optimizer, and one density-core gravity pass at strength 0.7. The quotient layout is independent of UMAP coordinates; UMAP appears only as a reference panel. Labels are used only for colors and evaluation.

After gravity was added to the estimator API, the research notebook now sets `gravity_mode=None` for each base fit and applies gravity explicitly afterward. This keeps the separate second phase from silently running an additional Mexican-hat force inside the fit.


| 640k layout | Total time | Trustworthiness | Input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| 3-MST IMSTE baseline | 48.9 s | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Quotient FR raw weights + gravity 0.7 | 42.6 s | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| Quotient FR square-root weights + gravity 0.7 | 39.8 s | 0.8798 | 0.0271 | 0.8145 | 0.4486 | 0.1265 | 0.000863 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

Both quotient layouts preserve more input neighbors and improve trustworthiness and domain-neighborhood purity over the independent IMSTE baseline, while cutting relative local radius by about 54% (raw weights) and 49% (square-root weights). The square-root layout slightly improves trustworthiness, overlap, and pointwise domain purity over raw weights, but loses label silhouette and macro purity; its normalized local radius is also a little larger than raw weights. Before centroid tethering, raw-weight quotient FR + IMSTE + density-core gravity was the best-balanced configuration; the updated selection is documented below. Square-root weights remain a visual option for more distinct regions, with the measured label-separation tradeoff shown above.

The raw and square-root candidates took 42.6s and 39.8s including the shared quotient preparation and gravity pass; the independent baseline took 48.9s in the same current session. The one-time Leiden graph build was shared across quotient variants. UMAP's 308.8s is the saved reference fit. The fixed renders show substantial improvement over the broad baseline fan, with some tails and separated small groups still visible.

## Rejected full-size spectral quotient layout

I tested normalized-Laplacian eigenvectors of the same 21-region Leiden quotient graph as an alternative to the Fruchterman–Reingold placement. Each region received the same small within-region SVD offsets, followed by the same three-MST IMSTE fit and one density-core gravity pass. This initialization is independent of UMAP.


| 640k layout | Time | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| 3-MST IMSTE baseline | 48.9 s | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Raw-weight quotient FR + gravity 0.7 | 42.6 s | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| Spectral quotient + gravity 0.7 | 40.7 s | 0.8411 | 0.0198 | 0.7944 | 0.3922 | 0.0989 | 0.000591 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

The spectral layout has the smallest relative local radius, but its fixed render still has a broad, connected fan and its trustworthiness, overlap, domain purities, and silhouette are all below the raw-weight quotient-FR candidate. Radius alone is therefore not a useful compactness win here: the spectral layout contracts local neighborhoods without forming better separated semantic regions. I reject it; the later centroid-tether screen builds on raw-weight quotient FR + IMSTE + one density-core gravity pass. The remaining tails and bridges are still an open limitation.

## Rejected higher-resolution quotient layout

I repeated the quotient-FR experiment with Leiden resolution 3.0, which produced 90 regions, then used the same per-region SVD offsets, independent three-MST IMSTE fit, and one density-core gravity pass. The 21-region result at resolution 0.8 is the comparison candidate. The shared RINND neighbor graph, Leiden partition, quotient layout, and local offsets took about 98 seconds; IMSTE took 34.9 seconds and the gravity pass 0.2 seconds. Total first-run time was 133.2 seconds, versus 42.6 seconds for the 21-region candidate and 48.9 seconds for the independent baseline.


| 640k layout | Total time | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| 3-MST IMSTE baseline | 48.9 s | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| 21-region quotient FR + gravity 0.7 | 42.6 s | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| 90-region quotient FR + gravity 0.7 | 133.2 s | 0.8409 | 0.0408 | 0.8255 | 0.4409 | 0.0993 | 0.001282 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

The 90-region layout gains input-neighbor overlap and pointwise domain purity, but its local radius is larger and macro purity and silhouette fall well below the 21-region layout. The fixed render has separated lobes but still spreads them across a broad shape. Its first-run setup is also about three times slower than the 21-region path. I reject resolution 3.0 for the default compactness configuration; resolution 0.8 remains the better-balanced choice.

## What the optimizer changes, and full-size alternatives

All comparisons below use the same 640k subset, 512 fixed anchors, exact input 15-NN ranks, label scoring protocol, and the same raw-weight 21-region quotient-FR initialization where applicable. Each fit explicitly disables internal gravity and applies one density-core phase after IMSTE. UMAP coordinates are used only as a reference.


| Stage/layout | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|
| Quotient-FR initialization | 0.8886 | 0.0056 | 0.7948 | 0.4764 | 0.1880 | 0.001238 |
| After 3-MST IMSTE | 0.8479 | 0.0148 | 0.8003 | 0.4247 | 0.1753 | 0.004183 |
| After one density-core phase, strength 0.7 | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| Independent IMSTE baseline | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| UMAP reference | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

This separates the stages: the quotient start is compact and has high trustworthiness and silhouette, but its exact input-neighbor overlap is weak. Three-MST refinement raises that overlap, while expanding relative local radius more than threefold and reducing silhouette. The gravity pass then contracts the regions and improves all measured neighborhood scores over the independent baseline. The remaining fan is therefore tied to the interaction between the sparse MST objective and the independent start; changing initialization alone does not preserve both compactness and input-neighbor fidelity.

### Local-neighbor attraction and MST-count screens




| Variant | Total time | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| Selected: 3 MST + density-core gravity 0.7 | 42.6 s | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| 3 MST + local kNN attraction + gravity, 100 epochs | 241.4 s | 0.8751 | 0.0191 | 0.8130 | 0.4799 | 0.2077 | 0.000715 |
| 3 MST + local kNN attraction + gravity, 20 epochs | 89.9 s | 0.8764 | 0.0121 | 0.8105 | 0.4626 | 0.2176 | 0.000644 |
| 3 MST + gravity, 20 epochs | 41.6 s | 0.8690 | 0.0100 | 0.8003 | 0.4863 | 0.2084 | 0.000611 |
| 2 MST + gravity 0.7 | 42.4 s | 0.8709 | 0.0249 | 0.8156 | 0.4404 | 0.1827 | 0.000756 |
| 1 MST + gravity 0.7 | 43.6 s | 0.8602 | 0.0497 | 0.7974 | 0.5231 | 0.1600 | 0.000503 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

Adding all local-neighbor attraction edges slightly raises trustworthiness, pointwise purity, silhouette, and compactness at 100 epochs, but lowers exact input-neighbor overlap and macro purity while taking 5.7 times as long as the selected candidate. At 20 epochs it is faster but overlap and macro purity fall further. Neither is a balanced upgrade. The 20-epoch MST-only run is also compact but loses too much exact overlap. One MST improves overlap and macro purity and contracts the local radius, but the fixed render is fragmented, and trustworthiness, domain purity, and silhouette fall; it does not yield coherent regions. The raw-weight 21-region, three-MST candidate was the best measured balance before centroid tethering. Its residual tails and bridges mean the fan is reduced rather than eliminated.

## Repulsion and local Mexican-hat force screens

I varied optimizer repulsion on the same raw quotient-FR start and also applied the local Mexican-hat second phase after the same 3-MST fit. The Mexican-hat pass used eight current-layout neighbors, attraction range 1.0, near-repulsion range 0.2, step size 0.05, and three iterations. Both candidates use exactly one gravity phase after the base IMSTE fit.



| Variant | Total time | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| Selected: repulsion 0.05 + density-core gravity 0.7 | 42.6 s | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| Repulsion 0.01 + density-core gravity 0.7 | 72.6 s | 0.8776 | 0.0250 | 0.8098 | 0.4925 | 0.2219 | 0.000734 |
| Repulsion 0.05 + local Mexican-hat gravity | 44.8 s | 0.8739 | 0.0264 | 0.8117 | 0.5063 | 0.1883 | 0.000775 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

Lowering `lambda_rep` to 0.01 raises silhouette and slightly lowers relative radius, but overlap, domain purity, and macro purity fall, and this run takes longer. The broad fan remains in the fixed rendering. Adding a very local Mexican-hat force changes the scores only marginally and does not visibly reshape the layout. Keep the selected density-core second phase and `lambda_rep=0.05`; stronger local force tuning is not supported by this full-size comparison.

## Updated candidate: partial quotient-centroid tether

The stage diagnostic showed that three-MST refinement moves the 21 Leiden-region centers away from their quotient-FR initialization. I tested a label-independent post-fit correction: for every point in region r, translate its refined coordinate by `alpha * (c_init[r] - c_fit[r])`, where `c_init` and `c_fit` are the mean region coordinates before and after IMSTE. This preserves pairwise offsets within each region before the usual single density-core phase. No UMAP coordinates or labels enter the transform.


| 640k layout | Total time | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| Independent IMSTE baseline | 48.9 s | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Quotient FR + 3 MST + gravity, no tether | 42.6 s | 0.8739 | 0.0262 | 0.8120 | 0.5005 | 0.1883 | 0.000766 |
| Centroid tether, alpha 0.25 | 42.7 s | 0.8759 | 0.0267 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| Centroid tether, alpha 0.5 | 42.8 s | 0.8768 | 0.0270 | 0.8133 | 0.5279 | 0.2192 | 0.001004 |
| Centroid tether, alpha 0.75 | 42.7 s | 0.8769 | 0.0266 | 0.8113 | 0.5184 | 0.2316 | 0.001154 |
| Centroid tether, alpha 1.0 | 42.7 s | 0.8768 | 0.0257 | 0.8095 | 0.5011 | 0.2382 | 0.001346 |
| UMAP reference | 308.8 s | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

Alpha 0.25 is the current balanced choice: it improves every neighborhood and label-separation score over the no-tether quotient candidate, while its relative local radius remains 48% below the independent baseline. The transform adds negligible time. Larger tether values raise silhouette and macro purity in places but also expand the local radius; alpha 0.5 is an alternate when stronger separation matters more than the tightest radius. The rendering shows better-defined regions, though the global fan and sparse tails are still visible. This is the best measured configuration so far, not a complete removal of the fan.

## Corrected uniform-weight quotient screen

The first saved uniform-weight quotient runs did not disable the estimator’s in-fit gravity default. I reran this candidate from its saved initialization with `gravity_mode=None`, then applied one density-core phase (strength 0.7, resolution 0.8, core quantile 0.95). This keeps the three-MST optimizer, 21 RINND/Leiden regions, 640k subset, seed, and scoring protocol fixed. No UMAP coordinates were used for initialization.


| Layout | Total time (s) | Trustworthiness | Input 15-NN overlap | Domain purity | Macro purity | Silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| Independent IMSTE baseline | 48.9 | 0.8401 | 0.0195 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Uniform quotient FR + gravity | 55.8 | 0.8284 | 0.0206 | 0.7595 | 0.4519 | -0.2450 | 0.001126 |
| Uniform quotient FR + centroid tether (α=0.25) + gravity | 55.8 | 0.8375 | 0.0219 | 0.7814 | 0.4676 | -0.2425 | 0.001043 |
| Selected raw-weight quotient FR + tether (α=0.25) + gravity | 42.7 | 0.8759 | 0.0267 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| UMAP reference | 308.8 | 0.8933 | 0.0772 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

Uniform edge weights preserve a similar fan in the fixed render. Although the relative local radius is smaller than the baseline, the embedding loses trustworthiness and pointwise domain purity, and its label silhouette becomes negative. The centroid tether recovers a little fidelity but remains clearly below the selected raw-weight candidate on every neighborhood and silhouette measure, while running longer. **Reject uniform quotient weights; retain raw weights with the partial centroid tether as the current leading candidate. The global fan remains unresolved.**

## Density-core quantile screen

Starting from the same raw-weight quotient initialization and three-MST fit, I applied the α=0.25 region-centroid tether and varied only the density-core quantile. Each version used one post-fit density-core pass at strength 0.7 and resolution 0.8. Labels were used only for scores and color.


| Core quantile | Time (s) | Trustworthiness | Input 15-NN overlap | Domain purity | Macro purity | Silhouette | Relative local radius |
|---:|---:|---:|---:|---:|---:|---:|---:|
| 0.95 | 42.8 | 0.8759 | 0.02669 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| 0.98 | 42.8 | 0.8764 | 0.02604 | 0.8132 | 0.5150 | 0.2020 | 0.000861 |
| 0.99 | 42.8 | 0.8765 | 0.02591 | 0.8151 | 0.5225 | 0.2035 | 0.000856 |
| 0.995 | 42.8 | 0.8760 | 0.02643 | 0.8116 | 0.4962 | 0.2057 | 0.000843 |
| 0.999 | 42.8 | 0.8761 | 0.02630 | 0.8116 | 0.5086 | 0.2041 | 0.000824 |

Moving from 0.95 to 0.99 slightly improves trustworthiness, pointwise and macro purity, silhouette, and normalized local radius; overlap drops by 0.0008. Moving to 0.995 or 0.999 tightens local radius further but reduces macro purity, and the render remains fan-shaped at every threshold. This is a small score tradeoff, not a solution to the global shape. Keep 0.95 as the balanced default for now; 0.99 is a reasonable macro-purity variant to revisit with a layout change.

## Inter-region attraction diagnostic

The intra-region density-core phase compacts points inside each Leiden region but leaves their global fan arrangement. I tested whether nearby high-dimensional region means could gently pull whole regions together. For each region, I connected its three nearest region means in the original input feature space, weighted neighbors by distance and region mass, and translated each region as a unit for one synchronous step. No labels or UMAP coordinates entered the force.


| Inter-region pull | Trustworthiness | Input 15-NN overlap | Domain purity | Macro purity | Silhouette | Relative local radius |
|---:|---:|---:|---:|---:|---:|---:|
| 0 (selected α=0.25 tether) | 0.8759 | 0.02669 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| 0.05 | 0.8760 | 0.02578 | 0.8142 | 0.5043 | 0.2039 | 0.000891 |
| 0.15 | 0.8753 | 0.02565 | 0.8094 | 0.5081 | 0.2061 | 0.000922 |
| 0.30 | 0.8746 | 0.02578 | 0.8090 | 0.5006 | 0.2071 | 0.000954 |
| 0.50 | 0.8721 | 0.02487 | 0.8060 | 0.4574 | 0.2050 | 0.000981 |

The neighborhood-based pull barely changes the displayed shape. Increasing it mildly raises silhouette but progressively lowers overlap and compactness; macro purity drops sharply at 0.5. Reject this one-step feature-centroid force. The remaining fan cannot be fixed by stronger intra-region core selection or this simple inter-region smoothing, so the next candidate needs to change how the region centers are initialized or optimized. The extra feature-centroid setup time is not included in the selected IMSTE fit time.

## Rejected full-size Kamada–Kawai quotient layout

To test whether the fan came from the Fruchterman–Reingold arrangement of the 21 Leiden regions, I rebuilt the same weighted RINND quotient graph and changed only its 2D center layout. One Kamada–Kawai variant used inverse normalized edge affinity as target edge length; the other ignored edge weights. Both used the same per-region PCA offsets, three-MST IMSTE fit, one density-core pass, seed, and 640k subset. No UMAP coordinates were used.


| Layout | Total time (s) | Trustworthiness | Input 15-NN overlap | Domain purity | Macro purity | Silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| Selected raw-weight FR + tether α=0.25 | 42.8 | 0.8759 | 0.02669 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| Kamada–Kawai, inverse affinity lengths | 56.7 | 0.8687 | 0.02122 | 0.8147 | 0.5275 | 0.0397 | 0.000803 |
| Kamada–Kawai, unweighted | 55.7 | 0.8459 | 0.01901 | 0.7701 | 0.4199 | -0.1223 | 0.001114 |
| UMAP reference | 308.8 | 0.8933 | 0.07721 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

Kamada–Kawai changes the arrangement but leaves a broad, fragmented fan. The weighted version has a smaller relative local radius and slightly higher pointwise and macro domain purity, but loses trustworthiness, exact input-neighbor overlap, and most label silhouette while taking longer. The unweighted version performs worse. Reject both; the remaining global shape gap persists across FR, spectral, and Kamada–Kawai quotient layouts. The raw-weight FR initialization with α=0.25 centroid tether remains the best-balanced candidate found so far, but it does not yet meet the visual goal.

## Rejected coarser density-region partition

I tested whether merging Leiden regions would absorb the fan tails. Starting from the same raw-weight quotient FR + three-MST fit and α=0.25 centroid tether, I changed only the Leiden resolution used by the density-core second phase from 0.8 (21 regions) to 0.2 (5 regions). The cached RINND/Leiden graph and the exact fixed-anchor evaluation protocol were used; labels were excluded from graph and force construction.


| Layout | Regions | Cold total time estimate | Trustworthiness | Input 15-NN overlap | Domain purity | Macro purity | Silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|---:|
| Independent IMSTE baseline | — | 48.9 s | 0.8401 | 0.01953 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Density-core resolution 0.2 | 5 | 149.5 s | 0.8594 | 0.01823 | 0.8034 | 0.4517 | 0.0535 | 0.000701 |
| Selected density-core resolution 0.8 + centroid tether | 21 | 153.4 s | 0.8759 | 0.02669 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| UMAP reference | — | 308.8 s | 0.8933 | 0.07721 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

The coarser partition is slightly more compact by relative local radius, but trustworthiness, exact neighbor overlap, and silhouette fall substantially. The fixed render shows five broad lobes with the fan/tails still visible. It also requires essentially the same cold graph-build cost as resolution 0.8. Reject resolution 0.2; retain 0.8. This screen changes only the post-fit region partition, not the three-MST initialization.

## Region-center packing screen: ring versus compact disk

The previous screen kept the 21 Leiden regions fixed internally and tested whether rearranging only their centers could remove the fan. Region membership came from the feature-space RINND/Leiden graph; labels and UMAP coordinates were excluded from every transformation. The ring variant orders region centers by their polar angle in the existing IMSTE layout, then translates each region as a rigid group to equally spaced positions on a circle. Its radius uses the sum of the region 90%-mass radii plus a gap of 0.25 times the median radius. The disk variant instead minimizes packed-center spread with soft non-overlap constraints. Both start from raw-weight quotient FR + three-MST IMSTE + density-core gravity 0.7 + α=0.25 centroid tether.



| 640k layout | Fit time | Trustworthiness | Exact input 15-NN overlap | Domain purity | Macro purity | Label silhouette | Relative local radius |
|---|---:|---:|---:|---:|---:|---:|---:|
| Independent IMSTE baseline | 48.9 s | 0.8401 | 0.01953 | 0.7999 | 0.3950 | 0.1456 | 0.001675 |
| Quotient FR + tether α=0.25 (current candidate) | 42.8 s | 0.8759 | 0.02669 | 0.8124 | 0.5091 | 0.2028 | 0.000877 |
| Ring pack α=1.0 | 42.8 s | 0.8698 | 0.02695 | 0.8060 | 0.4935 | 0.0911 | 0.000710 |
| Disk pack α=1.0 | 42.8 s | 0.8528 | 0.02773 | 0.8089 | 0.5127 | -0.0618 | 0.001205 |
| UMAP reference | 308.8 s | 0.8933 | 0.07721 | 0.8430 | 0.5633 | 0.0747 | 0.003605 |

The full ring translation removes the broad fan and produces separated, locally coherent regions. Trustworthiness falls by 0.0061 and pointwise domain purity by 0.0064 relative to the current candidate; exact input-neighbor overlap is unchanged within 0.0003, and macro purity falls by 0.0155. Its relative local radius is 19% smaller. Its label silhouette falls to 0.0911, still above the UMAP reference's 0.0747, but well below the current candidate's 0.2028. Reapplying the estimator utility `pack_region_centers_on_ring` to the cached 640k gravity output reproduces these metrics exactly; its maximum within-region coordinate difference is 1.2e-7 from float32 rounding. Packing takes 0.10 s for 640,000 rows across 21 regions. The 42.8 s remains the IMSTE fit time. It changes long-range region arrangement and should be treated as a compact visualization option rather than a preservation of global distances.

The disk optimizer stopped at its evaluation limit with a maximum 90%-radius overlap of 0.0009. The disk pack reduces the overall center spread further but creates overlap and mixing: silhouette becomes negative and trustworthiness drops by 0.0231. Reject that packing. For this fixed arXiv sample, ring packing is the stronger compact-view alternative; retain the α=0.25 tethered embedding as the fidelity-first candidate. The estimator exposes ring packing as an opt-in second-phase setting. Validation on another dataset remains open before making it a general default.